In [ ]:
%pip install -U gradio

In [ ]:
import gradio as gr
print("Installed Gradio version:", gr.__version__)

Installed Gradio version: 6.30.0


In [ ]:
# ============================================================
# AFTER HOURS — CHAT ROOM
# Google Colab | Gradio 6.x
# ============================================================

import gradio as gr
import threading
import tempfile
import os
import shutil
import uuid
from datetime import datetime, timezone
from zoneinfo import ZoneInfo
from html import escape
from urllib.parse import quote

# ============================================================
# CONFIGURATION
# ============================================================

messages = []
joined_users = set()
room_lock = threading.Lock()

MEDIA_DIR = os.path.join(tempfile.gettempdir(), "after_hours_media")
os.makedirs(MEDIA_DIR, exist_ok=True)

MAX_FILE_SIZE = 5 * 1024 * 1024
MAX_FILES = 5

TIMEZONES = [
    ("India (IST)", "Asia/Kolkata"),
    ("United Kingdom", "Europe/London"),
    ("New York", "America/New_York"),
    ("Los Angeles", "America/Los_Angeles"),
    ("Singapore", "Asia/Singapore"),
    ("Tokyo", "Asia/Tokyo"),
    ("Sydney", "Australia/Sydney"),
    ("Berlin", "Europe/Berlin"),
    ("UTC", "UTC"),
]

EMOJIS = [
    "😀", "😂", "🥹", "😍", "🥰", "😎",
    "😭", "🤔", "😴", "🙄", "😏", "🤭",
    "🤗", "😈", "❤️", "🖤", "💀", "✨",
    "🔥", "🎬", "🍿", "🌙", "⭐", "💌",
    "💔", "🫶", "👏", "👍", "👀", "🥳",
    "🤍", "☕", "💯", "🎉", "🙏", "😊",
]

# ============================================================
# HELPERS
# ============================================================

def get_timezone(selection):
    try:
        ZoneInfo(selection)
        return selection
    except Exception:
        return "Asia/Kolkata"


def now_utc():
    return datetime.now(timezone.utc).isoformat()


def format_time(timestamp, zone):
    try:
        return datetime.fromisoformat(timestamp).astimezone(
            ZoneInfo(get_timezone(zone))
        ).strftime("%d %b %Y, %I:%M %p")
    except Exception:
        return "--:--"


def media_url(path):
    return "/gradio_api/file=" + quote(
        os.path.abspath(path), safe="/"
    )


# ============================================================
# CHAT RENDERING
# ============================================================

def render_chat(zone="Asia/Kolkata"):
    with room_lock:
        snapshot = list(messages)

    if not snapshot:
        return """
        <div class="empty-chat">
            <div class="empty-icon">🍿</div>
            <div class="empty-title">Welcome to After Hours</div>
            <div class="empty-subtitle">
                Your late-night conversations start here.
            </div>
        </div>
        """

    output = ['<div class="chat-feed">']

    for msg in snapshot:
        name = escape(msg.get("name", "User"))
        timestamp = escape(format_time(msg["time"], zone))
        kind = msg["kind"]

        if kind == "join":
            output.append(
                '<div class="join-event">'
                f'<b>{name}</b> joined After Hours'
                f'<span class="event-time">{timestamp}</span>'
                '</div>'
            )
            continue

        output.append('<div class="chat-message">')
        output.append(
            '<div class="message-heading">'
            f'<span class="sender">{name}</span>'
            f'<span class="timestamp">{timestamp}</span>'
            '</div>'
        )

        if kind == "message":
            text = escape(msg.get("text", "")).replace("\n", "<br>")
            output.append(
                f'<div class="message-bubble">{text}</div>'
            )

        elif kind == "image":
            url = escape(media_url(msg["path"]), quote=True)
            filename = escape(msg["filename"])

            output.append(
                '<div class="media-card">'
                f'<a href="{url}" target="_blank" '
                'rel="noopener noreferrer">'
                f'<img class="chat-image" src="{url}" '
                f'alt="{filename}" loading="lazy">'
                '</a>'
                f'<div class="media-caption">{filename}</div>'
                '</div>'
            )

        elif kind == "pdf":
            url = escape(media_url(msg["path"]), quote=True)
            filename = escape(msg["filename"])
            size_kb = msg.get("size", 0) / 1024

            output.append(
                '<div class="media-card pdf-card">'
                '<div class="pdf-icon">PDF</div>'
                '<div class="pdf-details">'
                f'<div class="pdf-name">{filename}</div>'
                f'<div class="pdf-meta">{size_kb:.1f} KB</div>'
                '</div>'
                f'<a class="pdf-download" href="{url}" '
                f'download="{filename}">Download</a>'
                '</div>'
            )

        output.append('</div>')

    output.append('</div>')
    return "".join(output)


def render_alert(zone):
    with room_lock:
        latest = next(
            (m for m in reversed(messages) if m["kind"] == "join"),
            None
        )

    if latest:
        name = escape(latest["name"])
        return (
            f"**{name} joined the chat** · "
            f"{format_time(latest['time'], zone)}"
        )

    return "🍿 Your room is ready."


def refresh_chat(selection):
    zone = get_timezone(selection)
    return render_chat(zone), render_alert(zone)


# ============================================================
# JOIN ROOM
# ============================================================

def join_room(name, selection):
    name = (name or "").strip()
    zone = get_timezone(selection)

    if not name:
        return (
            render_chat(zone),
            render_alert(zone),
            "Enter a display name to join.",
            gr.update(interactive=True),
            False,
        )

    with room_lock:
        if name.casefold() not in joined_users:
            joined_users.add(name.casefold())
            messages.append({
                "kind": "join",
                "name": name,
                "time": now_utc(),
            })

    return (
        render_chat(zone),
        render_alert(zone),
        f"Welcome to After Hours, {name}!",
        gr.update(interactive=False),
        True,
    )


# ============================================================
# SEND TEXT MESSAGE
# ============================================================

def send_message(name, text, joined, selection):
    zone = get_timezone(selection)
    name = (name or "").strip()
    text = (text or "").strip()

    if not joined:
        return render_chat(zone), text, "Join the room before sending."

    if not text:
        return render_chat(zone), "", ""

    with room_lock:
        messages.append({
            "kind": "message",
            "name": name,
            "text": text,
            "time": now_utc(),
        })

    return render_chat(zone), "", ""


# ============================================================
# IMAGE AND PDF SHARING
# ============================================================

def share_media(name, files, joined, selection, expected_kind):
    zone = get_timezone(selection)
    name = (name or "").strip()

    if not joined:
        return render_chat(zone), [], "Join the room before sharing files."

    if not files:
        return render_chat(zone), [], "Choose a file first."

    if not isinstance(files, list):
        files = [files]

    if len(files) > MAX_FILES:
        return render_chat(zone), files, "Select no more than 5 files."

    count = 0
    errors = []

    for item in files:
        source = str(item)

        if not os.path.isfile(source):
            errors.append("A selected file could not be read.")
            continue

        filename = os.path.basename(source)
        ext = os.path.splitext(filename)[1].lower()
        size = os.path.getsize(source)

        if size > MAX_FILE_SIZE:
            errors.append(f"{filename}: exceeds the 5 MB limit.")
            continue

        try:
            with open(source, "rb") as file:
                header = file.read(16)
        except OSError:
            errors.append(f"{filename}: could not be read.")
            continue

        if expected_kind == "pdf":
            if ext != ".pdf" or not header.startswith(b"%PDF-"):
                errors.append(f"{filename}: select a valid PDF.")
                continue
            kind = "pdf"

        else:
            if ext not in (".png", ".jpg", ".jpeg", ".gif", ".webp"):
                errors.append(f"{filename}: unsupported image format.")
                continue

            valid_image = (
                header.startswith(b"\x89PNG\r\n\x1a\n")
                or header.startswith(b"\xff\xd8\xff")
                or header.startswith((b"GIF87a", b"GIF89a"))
                or (
                    len(header) >= 12
                    and header[:4] == b"RIFF"
                    and header[8:12] == b"WEBP"
                )
            )

            if not valid_image:
                errors.append(f"{filename}: invalid image file.")
                continue

            kind = "image"

        destination = os.path.join(
            MEDIA_DIR, uuid.uuid4().hex + ext
        )

        try:
            shutil.copy2(source, destination)
        except OSError:
            errors.append(f"{filename}: could not save.")
            continue

        with room_lock:
            messages.append({
                "kind": kind,
                "name": name,
                "filename": filename,
                "path": destination,
                "size": size,
                "time": now_utc(),
            })

        count += 1

    if errors:
        status = f"Shared {count} file(s). " + " ".join(errors)
    elif count:
        status = f"Successfully shared {count} file(s)."
    else:
        status = "No files were shared."

    return render_chat(zone), [], status


# ============================================================
# DOWNLOAD CHAT
# ============================================================

def download_chat(selection):
    zone = get_timezone(selection)

    with room_lock:
        snapshot = list(messages)

    if not snapshot:
        raise gr.Error("There are no messages to download yet.")

    lines = [
        "AFTER HOURS — CHAT HISTORY",
        "=" * 40,
        f"Exported: {format_time(now_utc(), zone)}",
        f"Time zone: {zone}",
        "",
    ]

    for msg in snapshot:
        timestamp = format_time(msg["time"], zone)
        name = msg.get("name", "User")

        if msg["kind"] == "join":
            line = f"[{timestamp}] {name} joined the chat"

        elif msg["kind"] in ("image", "pdf"):
            line = (
                f"[{timestamp}] {name} shared "
                f"{msg['kind']}: {msg['filename']}"
            )

        else:
            line = f"[{timestamp}] {name}: {msg['text']}"

        lines.extend([line, ""])

    with tempfile.NamedTemporaryFile(
        mode="w",
        encoding="utf-8",
        suffix=".txt",
        prefix="after_hours_",
        delete=False,
    ) as file:
        file.write("\n".join(lines))
        return gr.update(value=file.name, visible=True)


# ============================================================
# MENU CONTROLS
# ============================================================

def toggle_attachment(is_open):
    new_state = not is_open
    return gr.update(visible=new_state), new_state


def select_upload_type(kind):
    if kind == "image":
        return gr.update(visible=True), gr.update(visible=False)

    return gr.update(visible=False), gr.update(visible=True)


def toggle_three_dot_menu(is_open):
    new_state = not is_open
    return gr.update(visible=new_state), new_state


def open_download_panel():
    return (
        gr.update(visible=False),
        False,
        gr.update(visible=True),
        gr.update(value=None, visible=False),
    )


# ============================================================
# EMOJI PICKER
# ============================================================

emoji_buttons = "".join(
    f'<button type="button" class="emoji-choice" '
    f'data-emoji="{escape(e, quote=True)}">{e}</button>'
    for e in EMOJIS
)

emoji_html = f"""
<div id="emoji-widget">
    <button type="button" id="emoji-toggle" aria-label="Choose emoji">
        <svg viewBox="0 0 24 24" aria-hidden="true">
            <circle cx="12" cy="12" r="9"></circle>
            <circle cx="9" cy="10" r="0.8" class="eye"></circle>
            <circle cx="15" cy="10" r="0.8" class="eye"></circle>
            <path d="M8 14c1 2 2.3 3 4 3s3-1 4-3"></path>
        </svg>
    </button>
    <div id="emoji-popup">
        <div class="popup-title">Choose an emoji</div>
        <div class="emoji-grid">{emoji_buttons}</div>
    </div>
</div>
"""


# ============================================================
# CSS — CHAT ROOM UI
# ============================================================

css = """
.gradio-container {
    width: min(calc(100% - 28px), 1100px) !important;
    max-width: 1100px !important;
    margin: auto !important;
    padding: 20px 0 28px !important;
    background: #0b141a !important;
    color: #e9edef !important;
}

.gradio-container, .gradio-container * {
    box-sizing: border-box !important;
}

.gradio-container .wrap {
    min-width: 0 !important;
}

#main-title {
    text-align: center;
}

#main-title h1 {
    color: #25d366 !important;
    font-size: clamp(30px, 4vw, 42px) !important;
    font-weight: 900 !important;
    margin: 0 0 6px !important;
}

#subtitle {
    color: #aebac1;
    text-align: center;
    margin: 0 0 22px;
}

/* Join room */

#join-row {
    align-items: end !important;
    gap: 12px !important;
}

#join-button, #join-button button {
    background: #128c7e !important;
    border-color: #128c7e !important;
    border-radius: 9px !important;
}

#join-alert {
    background: #172229;
    border: 1px solid #303d44;
    border-radius: 10px;
    padding: 9px 13px;
    margin: 10px 0 12px;
}

/* Chat header */

#chat-header {
    display: flex !important;
    flex-direction: row !important;
    align-items: center !important;
    justify-content: space-between !important;
    gap: 12px !important;
    padding: 12px 15px !important;
    margin-top: 8px !important;
    background: #202c33 !important;
    border: 1px solid #303d44 !important;
    border-radius: 13px 13px 0 0 !important;
}

.room-identity {
    display: flex;
    align-items: center;
    gap: 11px;
}

.room-avatar {
    width: 42px;
    height: 42px;
    border-radius: 50%;
    background: #128c7e;
    display: flex;
    align-items: center;
    justify-content: center;
    font-weight: 800;
    color: white;
}

.room-name {
    font-weight: 750;
}

.room-description {
    color: #aebac1;
    font-size: 12px;
    margin-top: 3px;
}

/* Three-dot menu */

#menu-column {
    flex: 0 0 42px !important;
    width: 42px !important;
    min-width: 42px !important;
}

#three-dot-button, #three-dot-button button {
    width: 40px !important;
    min-width: 40px !important;
    height: 40px !important;
    min-height: 40px !important;
    padding: 0 !important;
    background: transparent !important;
    color: #e9edef !important;
    border: 0 !important;
    box-shadow: none !important;
    font-size: 28px !important;
}

#three-dot-menu {
    width: fit-content !important;
    min-width: 0 !important;
    max-width: 230px !important;
    margin-left: auto !important;
    padding: 5px !important;
    background: #202c33 !important;
    border: 1px solid #34434a !important;
    border-radius: 10px !important;
}

#download-menu-item, #download-menu-item button {
    width: auto !important;
    min-width: 0 !important;
    max-width: 220px !important;
    min-height: 38px !important;
    padding: 8px 12px !important;
    margin: 0 !important;
    background: transparent !important;
    border: 0 !important;
    box-shadow: none !important;
    color: #e9edef !important;
    white-space: nowrap !important;
    text-align: left !important;
    font-size: 14px !important;
}

/* Chat history */

#chat-feed-panel {
    display: block !important;
    width: 100% !important;
    height: 380px !important;
    min-height: 260px !important;
    max-height: 55vh !important;
    overflow-y: auto !important;
    overflow-x: hidden !important;
    background: #0b141a !important;
    border: 1px solid #303d44 !important;
    border-top: 0 !important;
    border-radius: 0 0 13px 13px !important;
    padding: 15px !important;
}

#chat-feed-panel .chat-feed {
    display: flex;
    flex-direction: column;
    gap: 17px;
    width: 100%;
    min-width: 0;
}

.empty-chat {
    min-height: 245px;
    display: flex;
    flex-direction: column;
    align-items: center;
    justify-content: center;
    gap: 8px;
    text-align: center;
}

.empty-icon {
    font-size: 35px;
}

.empty-title {
    font-size: 21px;
    font-weight: 700;
}

.empty-subtitle {
    color: #8696a0;
    font-size: 14px;
}

#chat-feed-panel .chat-message {
    display: flex;
    flex-direction: column;
    align-items: flex-start;
    gap: 6px;
    min-width: 0;
    width: 100%;
}

.message-heading {
    display: flex;
    align-items: baseline;
    gap: 9px;
    flex-wrap: wrap;
}

.sender {
    font-size: 13px;
    font-weight: 700;
    color: #d8fdd2;
}

.timestamp, .event-time {
    color: #8696a0;
    font-size: 11px;
}

.message-bubble {
    display: inline-block;
    max-width: min(680px, 90%);
    padding: 10px 13px;
    background: #202c33;
    border-radius: 4px 13px 13px 13px;
    color: #e9edef;
    line-height: 1.5;
    overflow-wrap: anywhere;
    word-break: break-word;
}

.join-event {
    align-self: center;
    max-width: 100%;
    padding: 7px 12px;
    border: 1px solid #2d414b;
    border-radius: 20px;
    background: #17242c;
    color: #c6d2d9;
    text-align: center;
    font-size: 12px;
    overflow-wrap: anywhere;
}

.event-time {
    margin-left: 7px;
}

/* Image and document cards */

.media-card {
    max-width: min(320px, 100%);
    min-width: 0;
    padding: 7px;
    border: 1px solid #303d44;
    border-radius: 12px;
    background: #202c33;
    overflow: hidden;
}

#chat-feed-panel img.chat-image {
    display: block;
    width: 280px !important;
    height: 195px !important;
    max-width: 100% !important;
    object-fit: cover;
    border-radius: 8px;
}

.media-caption {
    padding: 8px 3px 2px;
    color: #d6dfe3;
    font-size: 12px;
    overflow-wrap: anywhere;
}

.pdf-card {
    display: flex;
    align-items: center;
    gap: 9px;
}

.pdf-icon {
    width: 40px;
    height: 40px;
    flex-shrink: 0;
    display: flex;
    align-items: center;
    justify-content: center;
    border-radius: 8px;
    background: #293e48;
    color: #ffaaaa;
    font-size: 11px;
    font-weight: 800;
}

.pdf-details {
    flex: 1;
    min-width: 0;
}

.pdf-name {
    font-size: 12px;
    font-weight: 600;
    overflow-wrap: anywhere;
}

.pdf-meta {
    font-size: 11px;
    color: #aebac1;
    margin-top: 4px;
}

.pdf-download {
    flex-shrink: 0;
    padding: 8px 9px;
    border-radius: 7px;
    background: #128c7e;
    color: white !important;
    text-decoration: none !important;
    font-size: 11px;
}

/* ============================================================
   COMPOSER — EMOJI + PLUS + MESSAGE + SEND
   ============================================================ */

#composer-row {
    display: flex !important;
    flex-direction: row !important;
    align-items: center !important;
    gap: 10px !important;
    margin-top: 14px !important;
    width: 100% !important;
}

/* Both icon columns have identical dimensions */

#emoji-column,
#attachment-column {
    flex: 0 0 54px !important;
    width: 54px !important;
    min-width: 54px !important;
    max-width: 54px !important;
    align-self: center !important;
}

#emoji-column > div,
#attachment-column > div {
    width: 54px !important;
    min-width: 54px !important;
    max-width: 54px !important;
}

/* Emoji widget */

#emoji-widget {
    position: relative !important;
    width: 54px !important;
    height: 54px !important;
    min-width: 54px !important;
    max-width: 54px !important;
    display: block !important;
}

#emoji-toggle {
    display: flex !important;
    align-items: center !important;
    justify-content: center !important;
    width: 54px !important;
    height: 54px !important;
    min-width: 54px !important;
    min-height: 54px !important;
    padding: 0 !important;
    border: 1px solid #35434a !important;
    border-radius: 16px !important;
    background: #202c33 !important;
    color: #e9edef !important;
    cursor: pointer !important;
}

#emoji-toggle svg {
    width: 26px !important;
    height: 26px !important;
    fill: none !important;
    stroke: #e9edef !important;
    stroke-width: 1.8 !important;
    stroke-linecap: round !important;
    stroke-linejoin: round !important;
}

/* FIX: visible PLUS button, matching emoji button */

#attachment-toggle,
#attachment-toggle button {
    display: flex !important;
    align-items: center !important;
    justify-content: center !important;
    width: 54px !important;
    height: 54px !important;
    min-width: 54px !important;
    max-width: 54px !important;
    min-height: 54px !important;
    max-height: 54px !important;
    padding: 0 !important;
    margin: 0 !important;
    border: 1px solid #35434a !important;
    border-radius: 16px !important;
    background: #202c33 !important;
    color: #e9edef !important;
    box-shadow: none !important;
    cursor: pointer !important;
    font-size: 0 !important;
}

/* Draw a proper plus symbol */

#attachment-toggle button::before {
    content: "+" !important;
    display: flex !important;
    align-items: center !important;
    justify-content: center !important;
    width: 100% !important;
    height: 100% !important;
    color: #e9edef !important;
    font-family: Arial, sans-serif !important;
    font-size: 34px !important;
    font-weight: 300 !important;
    line-height: 1 !important;
}

/* Message box */

#message-column {
    flex: 1 1 0 !important;
    min-width: 0 !important;
}

#message-input {
    margin: 0 !important;
    min-width: 0 !important;
}

#message-input label {
    color: #e9edef !important;
    font-size: 14px !important;
    margin-bottom: 5px !important;
}

#message-input textarea {
    min-height: 66px !important;
    max-height: 150px !important;
    padding: 12px 14px !important;
    background: #202c33 !important;
    color: #e9edef !important;
    border: 1px solid #35434a !important;
    border-radius: 14px !important;
    line-height: 1.45 !important;
    resize: vertical !important;
}

/* FIX: compact Send button; don't stretch vertically */

#send-column {
    flex: 0 0 100px !important;
    width: 100px !important;
    min-width: 100px !important;
    max-width: 100px !important;
    align-self: center !important;
}

#send-button,
#send-button button {
    display: flex !important;
    align-items: center !important;
    justify-content: center !important;
    width: 100% !important;
    min-width: 0 !important;
    height: 54px !important;
    min-height: 54px !important;
    max-height: 54px !important;
    padding: 0 12px !important;
    margin: 0 !important;
    border: 0 !important;
    border-radius: 14px !important;
    background: #128c7e !important;
    color: white !important;
    font-size: 15px !important;
    font-weight: 750 !important;
    white-space: nowrap !important;
    box-shadow: none !important;
}

/* Emoji popup */

#emoji-popup {
    display: none;
    position: absolute;
    bottom: 66px;
    left: 0;
    z-index: 10000;
    width: min(330px, calc(100vw - 36px));
    padding: 12px;
    border: 1px solid #35434a;
    border-radius: 15px;
    background: #172229;
    box-shadow: 0 12px 35px rgba(0, 0, 0, 0.4);
}

#emoji-popup.open {
    display: block !important;
}

.popup-title {
    font-size: 15px;
    font-weight: 750;
    margin-bottom: 10px;
    color: #e9edef;
}

.emoji-grid {
    display: grid;
    grid-template-columns: repeat(6, minmax(0, 1fr));
    gap: 6px;
}

.emoji-choice {
    display: flex;
    align-items: center;
    justify-content: center;
    min-width: 0;
    min-height: 39px;
    padding: 4px;
    border: 1px solid #35434a;
    border-radius: 9px;
    background: #202c33;
    font-size: 22px;
    cursor: pointer;
}

.emoji-choice:hover {
    background: #30434b;
}

/* Attachment menu */

#attachment-panel {
    width: fit-content !important;
    max-width: 100% !important;
    min-width: 0 !important;
    margin-top: 10px !important;
    padding: 13px !important;
    border: 1px solid #35434a !important;
    border-radius: 14px !important;
    background: #172229 !important;
}

#attachment-panel h3 {
    font-size: 15px !important;
    margin: 0 0 10px !important;
}

#attachment-options {
    display: flex !important;
    flex-direction: row !important;
    gap: 10px !important;
}

#image-option,
#document-option {
    flex: 0 1 160px !important;
    min-width: 120px !important;
}

#image-option button,
#document-option button {
    width: 100% !important;
    min-height: 44px !important;
    padding: 10px 13px !important;
    border: 1px solid #35434a !important;
    border-radius: 10px !important;
    background: #26343b !important;
    color: #e9edef !important;
    font-size: 14px !important;
    font-weight: 650 !important;
    white-space: nowrap !important;
}

/* Upload panels */

#image-upload-panel,
#document-upload-panel {
    margin-top: 12px !important;
    padding: 12px !important;
    border: 1px solid #35434a !important;
    border-radius: 11px !important;
    background: #202c33 !important;
}

#image-upload-panel button,
#document-upload-panel button {
    background: #128c7e !important;
    color: white !important;
    border-radius: 9px !important;
}

/* Download section */

#download-section {
    width: fit-content !important;
    max-width: 100% !important;
    min-width: 0 !important;
    margin: 12px 0 0 auto !important;
    padding: 8px !important;
    border: 1px solid #35434a !important;
    border-radius: 12px !important;
    background: #202c33 !important;
}

#download-button,
#download-button button {
    display: inline-flex !important;
    align-items: center !important;
    justify-content: center !important;
    width: auto !important;
    min-width: 0 !important;
    min-height: 38px !important;
    padding: 8px 12px !important;
    margin: 0 !important;
    border: 0 !important;
    border-radius: 8px !important;
    background: #374151 !important;
    color: white !important;
    white-space: nowrap !important;
    font-size: 14px !important;
}

#download-file {
    margin-top: 8px !important;
}

/* Status text */

#message-status {
    color: #aebac1 !important;
    font-size: 12px !important;
    margin-top: 5px !important;
}

/* Mobile layout */

@media (max-width: 600px) {
    .gradio-container {
        width: calc(100% - 16px) !important;
        padding-top: 12px !important;
    }

    #composer-row {
        gap: 6px !important;
    }

    #emoji-column,
    #attachment-column,
    #emoji-column > div,
    #attachment-column > div,
    #emoji-widget,
    #emoji-toggle,
    #attachment-toggle,
    #attachment-toggle button {
        flex-basis: 42px !important;
        width: 42px !important;
        min-width: 42px !important;
        max-width: 42px !important;
    }

    #emoji-widget,
    #emoji-toggle,
    #attachment-toggle,
    #attachment-toggle button {
        height: 44px !important;
        min-height: 44px !important;
        max-height: 44px !important;
    }

    #emoji-toggle {
        border-radius: 12px !important;
    }

    #attachment-toggle,
    #attachment-toggle button {
        border-radius: 12px !important;
    }

    #attachment-toggle button::before {
        font-size: 29px !important;
    }

    #emoji-toggle svg {
        width: 23px !important;
        height: 23px !important;
    }

    #send-column {
        flex-basis: 66px !important;
        width: 66px !important;
        min-width: 66px !important;
        max-width: 66px !important;
    }

    #send-button,
    #send-button button {
        height: 44px !important;
        min-height: 44px !important;
        max-height: 44px !important;
        padding: 0 5px !important;
        font-size: 11px !important;
        border-radius: 11px !important;
    }

    #message-input textarea {
        min-height: 58px !important;
        padding: 10px !important;
        font-size: 14px !important;
    }

    #chat-feed-panel {
        height: 340px !important;
        max-height: 50vh !important;
        padding: 10px !important;
    }

    #chat-feed-panel img.chat-image {
        width: 235px !important;
        height: 170px !important;
    }

    #attachment-panel {
        width: 100% !important;
    }

    #attachment-options {
        gap: 7px !important;
    }

    #image-option,
    #document-option {
        min-width: 0 !important;
        flex: 1 1 0 !important;
    }

    #image-option button,
    #document-option button {
        padding: 8px 5px !important;
        font-size: 12px !important;
    }

    .emoji-grid {
        grid-template-columns: repeat(6, minmax(0, 1fr));
    }
}
"""


# ============================================================
# BUILD APP
# ============================================================

with gr.Blocks(
    theme=gr.themes.Base(
        primary_hue="teal",
        neutral_hue="slate"
    ),
    css=css
) as app:

    gr.Markdown("# AFTER HOURS", elem_id="main-title")

    gr.HTML(
        '<div id="subtitle">'
        'Your late-night corner of the internet'
        '</div>'
    )

    with gr.Row(elem_id="join-row"):
        name_input = gr.Textbox(
            label="YOUR ALIAS",
            placeholder="Enter your name",
            scale=3
        )

        timezone_input = gr.Dropdown(
            choices=TIMEZONES,
            value="Asia/Kolkata",
            label="TIME ZONE",
            scale=2
        )

        join_button = gr.Button(
            "JOIN ROOM",
            variant="primary",
            elem_id="join-button",
            scale=1
        )

    alert = gr.Markdown(
        "🍿 Your room is ready.",
        elem_id="join-alert"
    )

    # Chat header and three-dot menu

    with gr.Row(elem_id="chat-header"):
        gr.HTML(
            '<div class="room-identity">'
            '<div class="room-avatar">AH</div>'
            '<div>'
            '<div class="room-name">After Hours</div>'
            '<div class="room-description">Group chat</div>'
            '</div></div>'
        )

        with gr.Column(
            elem_id="menu-column",
            scale=0,
            min_width=42
        ):
            three_dot_button = gr.Button(
                "⋮",
                elem_id="three-dot-button"
            )

    menu_state = gr.State(False)
    attachment_state = gr.State(False)
    joined_state = gr.State(False)

    with gr.Group(
        visible=False,
        elem_id="three-dot-menu"
    ) as three_dot_menu:
        download_menu_button = gr.Button(
            "↓  Download Chat",
            elem_id="download-menu-item"
        )

    chat_feed = gr.HTML(
        value=render_chat("Asia/Kolkata"),
        elem_id="chat-feed-panel"
    )

    # Composer: emoji, plus, text, send

    with gr.Row(
        elem_id="composer-row",
        equal_height=False
    ):

        with gr.Column(
            elem_id="emoji-column",
            scale=0,
            min_width=54
        ):
            gr.HTML(emoji_html)

        with gr.Column(
            elem_id="attachment-column",
            scale=0,
            min_width=54
        ):
            # The button text is supplied by CSS as a visible plus.
            attachment_button = gr.Button(
                "",
                elem_id="attachment-toggle"
            )

        with gr.Column(
            elem_id="message-column",
            scale=1,
            min_width=0
        ):
            message_input = gr.Textbox(
                label="YOUR MESSAGE",
                placeholder="Type your message...",
                elem_id="message-input",
                lines=2,
                max_lines=5
            )

        with gr.Column(
            elem_id="send-column",
            scale=0,
            min_width=100
        ):
            send_button = gr.Button(
                "SEND ➤",
                variant="primary",
                elem_id="send-button"
            )

    status = gr.Markdown("", elem_id="message-status")

    # Attachment options

    with gr.Group(
        visible=False,
        elem_id="attachment-panel"
    ) as attachment_panel:

        gr.Markdown("### Share a file")

        with gr.Row(elem_id="attachment-options"):
            image_option = gr.Button(
                "▧  Images",
                elem_id="image-option"
            )

            document_option = gr.Button(
                "▤  Documents",
                elem_id="document-option"
            )

        with gr.Group(
            visible=False,
            elem_id="image-upload-panel"
        ) as image_upload_panel:

            image_files = gr.File(
                label="Choose images",
                file_count="multiple",
                file_types=["image"],
                type="filepath"
            )

            send_images = gr.Button(
                "SEND IMAGES",
                variant="primary"
            )

            image_status = gr.Markdown("")

        with gr.Group(
            visible=False,
            elem_id="document-upload-panel"
        ) as document_upload_panel:

            document_files = gr.File(
                label="Choose PDF documents",
                file_count="multiple",
                file_types=[".pdf"],
                type="filepath"
            )

            send_documents = gr.Button(
                "SEND DOCUMENTS",
                variant="primary"
            )

            document_status = gr.Markdown("")

    # Download panel stays connected to the three-dot menu

    with gr.Group(
        visible=False,
        elem_id="download-section"
    ) as download_section:

        download_button = gr.Button(
            "↓  Download chat (.txt)",
            elem_id="download-button"
        )

        download_file = gr.File(
            label="",
            interactive=False,
            visible=False,
            elem_id="download-file"
        )

    # ========================================================
    # EVENTS
    # ========================================================

    join_button.click(
        join_room,
        inputs=[name_input, timezone_input],
        outputs=[
            chat_feed,
            alert,
            status,
            join_button,
            joined_state
        ]
    )

    send_button.click(
        send_message,
        inputs=[
            name_input,
            message_input,
            joined_state,
            timezone_input
        ],
        outputs=[chat_feed, message_input, status]
    )

    message_input.submit(
        send_message,
        inputs=[
            name_input,
            message_input,
            joined_state,
            timezone_input
        ],
        outputs=[chat_feed, message_input, status]
    )

    attachment_button.click(
        toggle_attachment,
        inputs=[attachment_state],
        outputs=[attachment_panel, attachment_state]
    )

    image_option.click(
        lambda: select_upload_type("image"),
        inputs=[],
        outputs=[image_upload_panel, document_upload_panel]
    )

    document_option.click(
        lambda: select_upload_type("document"),
        inputs=[],
        outputs=[image_upload_panel, document_upload_panel]
    )

    send_images.click(
        lambda name, files, joined, tz: share_media(
            name, files, joined, tz, "image"
        ),
        inputs=[
            name_input,
            image_files,
            joined_state,
            timezone_input
        ],
        outputs=[chat_feed, image_files, image_status]
    )

    send_documents.click(
        lambda name, files, joined, tz: share_media(
            name, files, joined, tz, "pdf"
        ),
        inputs=[
            name_input,
            document_files,
            joined_state,
            timezone_input
        ],
        outputs=[chat_feed, document_files, document_status]
    )

    three_dot_button.click(
        toggle_three_dot_menu,
        inputs=[menu_state],
        outputs=[three_dot_menu, menu_state]
    )

    download_menu_button.click(
        open_download_panel,
        inputs=[],
        outputs=[
            three_dot_menu,
            menu_state,
            download_section,
            download_file
        ]
    )

    download_button.click(
        download_chat,
        inputs=[timezone_input],
        outputs=[download_file]
    )

    timer = gr.Timer(2)

    timer.tick(
        refresh_chat,
        inputs=[timezone_input],
        outputs=[chat_feed, alert]
    )

    timezone_input.change(
        refresh_chat,
        inputs=[timezone_input],
        outputs=[chat_feed, alert]
    )

    # Emoji picker JavaScript

    app.load(
        fn=None,
        inputs=None,
        outputs=None,
        js="""
        () => {
            function installEmojiPicker() {
                const widget = document.querySelector("#emoji-widget");
                const popup = document.querySelector("#emoji-popup");
                const toggle = document.querySelector("#emoji-toggle");
                const textbox = document.querySelector(
                    "#message-input textarea"
                );

                if (!widget || !popup || !toggle || !textbox) return;
                if (widget.dataset.ready === "true") return;

                widget.dataset.ready = "true";

                toggle.addEventListener("click", event => {
                    event.preventDefault();
                    event.stopPropagation();
                    popup.classList.toggle("open");
                });

                popup.addEventListener("click", event => {
                    const button = event.target.closest(".emoji-choice");
                    if (!button) return;

                    event.preventDefault();
                    event.stopPropagation();

                    const start =
                        textbox.selectionStart ?? textbox.value.length;
                    const end =
                        textbox.selectionEnd ?? textbox.value.length;

                    textbox.setRangeText(
                        button.dataset.emoji,
                        start,
                        end,
                        "end"
                    );

                    textbox.dispatchEvent(
                        new Event("input", {bubbles: true})
                    );

                    textbox.focus();
                });

                document.addEventListener("click", event => {
                    if (!widget.contains(event.target)) {
                        popup.classList.remove("open");
                    }
                });
            }

            installEmojiPicker();
            setTimeout(installEmojiPicker, 500);
            setTimeout(installEmojiPicker, 1500);
        }
        """
    )


# ============================================================
# LAUNCH
# ============================================================

app.queue().launch(
    share=True,
    debug=True,
    allowed_paths=[MEDIA_DIR]
)

/tmp/ipykernel_3086/3853180245.py:1241: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://3122d67053d37c5e59.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://3122d67053d37c5e59.gradio.live
